# Load Dataset

In [64]:
%run capstone_EDA.ipynb

berhasil membaca file CSV.
jumlah data dan column: (4803, 20)
list Column: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity           

,Missing Count,Missing Percentage
homepage,3091,64.355611
tagline,844,17.572351
overview,3,0.062461
runtime,2,0.041641
release_date,1,0.020820


homepage dropped (if present).
Missing values after fill:
tagline         0
overview        0
runtime         0
release_date    0
dtype: int64


In [65]:
df_capstone.head()

,budget,id,original_language,original_title,overview,popularity,release_date,revenue,runtime,status,tagline,title,vote_average,vote_count,genres,keywords,production_companies,production_countries,spoken_languages
0,237000000,19995,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,2009-12-10,2787965087,162.0,Released,Enter the World of Pandora.,Avatar,7.2,11800,"Action, Adventure, Fantasy, Science Fiction","culture clash, future, space war, space colony...","Ingenious Film Partners, Twentieth Century Fox...","United States of America, United Kingdom","English, Español"
1,300000000,285,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,2007-05-19,961000000,169.0,Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"Adventure, Fantasy, Action","ocean, drug abuse, exotic island, east india t...","Walt Disney Pictures, Jerry Bruckheimer Films,...",United States of America,English
2,245000000,206647,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,2015-10-26,880674609,148.0,Released,A Plan No One Escapes,Spectre,6.3,4466,"Action, Adventure, Crime","spy, based on novel, secret agent, sequel, mi6...","Columbia Pictures, Danjaq, B24","United Kingdom, United States of America","Français, English, Español, Italiano, Deutsch"
3,250000000,49026,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,2012-07-16,1084939099,165.0,Released,The Legend Ends,The Dark Knight Rises,7.6,9106,"Action, Crime, Drama, Thriller","dc comics, crime fighter, terrorist, secret id...","Legendary Pictures, Warner Bros., DC Entertain...",United States of America,English
4,260000000,49529,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,2012-03-07,284139100,132.0,Released,"Lost in our world, found in another.",John Carter,6.1,2124,"Action, Adventure, Science Fiction","based on novel, mars, medallion, space travel,...",Walt Disney Pictures,United States of America,English


# Metadata Fusion — combine_feature

In [66]:
feature_cols = ['overview', 'genres', 'keywords', 'tagline']

# Isi missing value
for col in feature_cols:
    df_capstone[col] = df_capstone[col].fillna('')

# Gabungkan seluruh metadata teks

df_capstone['combine_feature'] = (
    df_capstone['overview'] + ' ' +
    df_capstone['genres'] + ' ' +
    df_capstone['keywords'] + ' ' +
    df_capstone['tagline']
)

print(df_capstone['combine_feature'].head())

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: combine_feature, dtype: object


# Text Preprocessing — NLP pipeline

In [67]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = text.split()

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]

    return ' '.join(tokens)


df_capstone['clean_feature'] = df_capstone['combine_feature'].apply(clean_text)

print(df_capstone['clean_feature'].head())

[nltk_data] Downloading package stopwords to C:\Users\faiz
[nltk_data]     hutabarat\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\faiz
[nltk_data]     hutabarat\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


0    nd century paraplegic marine dispatched moon p...
1    captain barbossa long believed dead come back ...
2    cryptic message bond past sends trail uncover ...
3    following death district attorney harvey dent ...
4    john carter warweary former military captain w...
Name: clean_feature, dtype: object


# TF-IDF Vectorization


In [68]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2),
    min_df=2
)

# Transform teks menjadi matriks TF-IDF

tfidf_matrix = vectorizer.fit_transform(df_capstone['clean_feature'])

print(tfidf_matrix.shape)

(4803, 5000)


# Cosine Similarity Matrix


In [69]:
from sklearn.metrics.pairwise import cosine_similarity

# =========================================
# COSINE SIMILARITY
# =========================================

cosine_sim_matrix = cosine_similarity(tfidf_matrix)

print(cosine_sim_matrix.shape)

(4803, 4803)


# Bayesian Weighted Rating

In [70]:
# Bayesian weighted rating + min-max normalization
try:
    # Pastikan data tersedia
    if 'vote_average' not in df_capstone.columns or 'vote_count' not in df_capstone.columns:
        raise KeyError('vote_average atau vote_count')

    df_capstone['vote_average'] = df_capstone['vote_average'].fillna(0).astype(float)
    df_capstone['vote_count'] = df_capstone['vote_count'].fillna(0).astype(float)

    # Parameter Bayesian smoothing
    C = df_capstone['vote_average'].mean()
    m = df_capstone['vote_count'].quantile(0.75)

    df_capstone['bayes_weighted_rating'] = (
        (df_capstone['vote_count'] / (df_capstone['vote_count'] + m)) * df_capstone['vote_average'] +
        (m / (df_capstone['vote_count'] + m)) * C
    )

    # Normalisasi ke [0, 1] menggunakan min-max scaling
    min_score = df_capstone['bayes_weighted_rating'].min()
    max_score = df_capstone['bayes_weighted_rating'].max()
    if max_score > min_score:
        df_capstone['bayes_weighted_rating_norm'] = (
            df_capstone['bayes_weighted_rating'] - min_score
        ) / (max_score - min_score)
    else:
        df_capstone['bayes_weighted_rating_norm'] = 0.0

    print('✓ Bayesian weighted rating berhasil dihitung.')
    print(f'  - C (average vote_average): {C:.4f}')
    print(f'  - m (75th percentile vote_count): {m:.1f}')
    print(f'  - Normalisasi range: [{df_capstone["bayes_weighted_rating_norm"].min():.4f}, {df_capstone["bayes_weighted_rating_norm"].max():.4f}]')
    display(df_capstone[['title', 'vote_average', 'vote_count', 'bayes_weighted_rating', 'bayes_weighted_rating_norm']].sort_values('bayes_weighted_rating_norm', ascending=False).head(5))
except NameError:
    print('✗ Variable df_capstone tidak ditemukan. Jalankan sel pemuatan data terlebih dahulu.')
except KeyError as e:
    print(f'✗ Kolom tidak ditemukan: {e}. Pastikan vote_average dan vote_count ada.')
except Exception as e:
    print(f'✗ Error: {e}')


✓ Bayesian weighted rating berhasil dihitung.
  - C (average vote_average): 6.0922
  - m (75th percentile vote_count): 737.0
  - Normalisasi range: [0.0000, 1.0000]


,title,vote_average,vote_count,bayes_weighted_rating,bayes_weighted_rating_norm
1881,The Shawshank Redemption,8.5,8205.0,8.301547,1.000000
3337,The Godfather,8.4,5893.0,8.143459,0.954675
662,Fight Club,8.3,9413.0,8.139688,0.953594
3232,Pulp Fiction,8.3,8428.0,8.122458,0.948654
65,The Dark Knight,8.2,12002.0,8.078054,0.935924


# Sentiment Analysis

In [71]:
!pip install textblob

In [72]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to C:\Users\faiz
[nltk_data]     hutabarat\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [73]:
# =========================================
# EMOTION-BASED SENTIMENT ANALYSIS
# =========================================

# Analisis sentimen berbasis keyword/emotion lexicon
# menggunakan overview film dari TMDB

# =========================================
# PREPROCESSING
# =========================================

df_capstone['keywords'] = (
    df_capstone['keywords']
    .fillna("")
    .str.lower()
)

# =========================================
# EMOTION LEXICON
# =========================================

emotion_lexicon = {

    "joy": [
        "love",
        "happy",
        "friendship",
        "funny",
        "family",
        "hero",
        "celebration",
        "adventure"
    ],

    "sadness": [
        "sad",
        "death",
        "loss",
        "pain",
        "cry",
        "lonely",
        "tragedy"
    ],

    "fear": [
        "horror",
        "ghost",
        "monster",
        "killer",
        "terror",
        "dark",
        "haunted"
    ],

    "anger": [
        "war",
        "revenge",
        "fight",
        "crime",
        "violence",
        "battle",
        "murder"
    ],

    "surprise": [
        "mystery",
        "magic",
        "alien",
        "future",
        "unexpected",
        "supernatural"
    ]
}

# =========================================
# FUNCTION EMOTION SCORE
# =========================================

def calculate_emotion(text):

    scores = {}

    for emotion, keywords in emotion_lexicon.items():

        score = 0

        for word in keywords:

            score += text.count(word)

        scores[emotion] = score

    return scores

# =========================================
# HITUNG EMOTION SCORE
# =========================================

emotion_scores = df_capstone[
    'keywords'
].apply(calculate_emotion)

emotion_df = pd.DataFrame(
    emotion_scores.tolist()
)

# Gabungkan ke dataframe utama
df_capstone = pd.concat(
    [df_capstone, emotion_df],
    axis=1
)

# =========================================
# DOMINANT EMOTION
# =========================================

emotion_columns = list(emotion_lexicon.keys())

def get_dominant_emotion(row):
    if row[emotion_columns].sum() == 0:
        return 'neutral'
    return row[emotion_columns].idxmax()

df_capstone['dominant_emotion'] = df_capstone.apply(
    get_dominant_emotion,
    axis=1
)

# =========================================
# SENTIMENT SCORE
# =========================================

df_capstone['sentiment_score'] = (
    df_capstone[emotion_columns]
    .sum(axis=1)
)

# =========================================
# HASIL
# =========================================

print(df_capstone[[
    'title',
    'dominant_emotion',
    'sentiment_score'
]].head())

                                      title dominant_emotion  sentiment_score
0                                    Avatar            anger                7
1  Pirates of the Caribbean: At World's End              joy                2
2                                   Spectre          neutral                0
3                     The Dark Knight Rises              joy                6
4                               John Carter         surprise                2


# Hybrid Scoring & Top-N Ranking

In [74]:
# =========================================
# HITUNG PARAMETER
# =========================================

C = df_capstone['vote_average'].mean()
m = df_capstone['vote_count'].quantile(0.90)

print("Global Average Rating (C):", round(C, 3))
print("Minimum Votes Required (m):", round(m, 3))

# =========================================
# FUNCTION BAYESIAN SCORE
# =========================================

def bayesian_weighted_rating(x, m=m, C=C):

    v = x['vote_count']
    R = x['vote_average']

    return (
        (v / (v + m) * R)
        +
        (m / (m + v) * C)
    )

# =========================================
# HITUNG SCORE                              <-- INI YANG KURANG
# =========================================

df_capstone['bayesian_score'] = (
    df_capstone.apply(
        bayesian_weighted_rating,
        axis=1
    )
)

# =========================================
# HYBRID SCORE
# =========================================

min_b = df_capstone['bayesian_score'].min()
max_b = df_capstone['bayesian_score'].max()

df_capstone['bayesian_norm'] = (
    df_capstone['bayesian_score'] - min_b
) / (max_b - min_b)

min_s = df_capstone['sentiment_score'].min()
max_s = df_capstone['sentiment_score'].max()

if max_s > min_s:
    df_capstone['sentiment_norm'] = (
        df_capstone['sentiment_score'] - min_s
    ) / (max_s - min_s)
else:
    df_capstone['sentiment_norm'] = 0.0

df_capstone['hybrid_score'] = (
    0.7 * df_capstone['bayesian_norm'] +
    0.3 * df_capstone['sentiment_norm']
)

print(df_capstone[[
    'title',
    'bayesian_score',
    'sentiment_score',
    'hybrid_score'
]].head())

Global Average Rating (C): 6.092
Minimum Votes Required (m): 1838.4
                                      title  bayesian_score  sentiment_score  \
0                                    Avatar        7.050669                7   
1  Pirates of the Caribbean: At World's End        6.665696                2   
2                                   Spectre        6.239396                0   
3                     The Dark Knight Rises        7.346721                6   
4                               John Carter        6.096368                2   

   hybrid_score  
0      0.631843  
1      0.414032  
2      0.261257  
3      0.678217  
4      0.276775  


# Fungsi Rekomendasi

In [75]:

# =========================================
# BAYESIAN WEIGHTED RATING (TMDB)
# =========================================

# R = rata-rata rating film
# v = jumlah vote film
# C = rata-rata vote keseluruhan dataset
# m = minimum vote agar dianggap populer

# Formula Bayesian:
# Weighted Rating = (v / (v + m) * R) + (m / (m + v) * C)

# =========================================
# HITUNG PARAMETER
# =========================================

C = df_capstone['vote_average'].mean()

m = df_capstone['vote_count'].quantile(0.90)

print("Global Average Rating (C):", round(C, 3))
print("Minimum Votes Required (m):", round(m, 3))

# =========================================
# FUNCTION BAYESIAN SCORE
# =========================================

def bayesian_weighted_rating(x, m=m, C=C):

    v = x['vote_count']
    R = x['vote_average']

    return (
        (v / (v + m) * R)
        +
        (m / (m + v) * C)
    )

# =========================================
# HITUNG SCORE
# =========================================

df_capstone['bayesian_score'] = (
    df_capstone.apply(
        bayesian_weighted_rating,
        axis=1
    )
)

print(df_capstone[[
    'title',
    'vote_average',
    'vote_count',
    'bayesian_score'
]].head())


Global Average Rating (C): 6.092
Minimum Votes Required (m): 1838.4
                                      title  vote_average  vote_count  \
0                                    Avatar           7.2     11800.0   
1  Pirates of the Caribbean: At World's End           6.9      4500.0   
2                                   Spectre           6.3      4466.0   
3                     The Dark Knight Rises           7.6      9106.0   
4                               John Carter           6.1      2124.0   

   bayesian_score  
0        7.050669  
1        6.665696  
2        6.239396  
3        7.346721  
4        6.096368  


# Simpan Model

In [76]:
import pickle

# Simpan dataset utama

df_capstone.to_csv('cleaned_data.csv', index=False)

# Simpan cosine similarity

with open('cosine_similarity.pkl', 'wb') as f:
    pickle.dump(cosine_sim_matrix, f)

print('Model berhasil disimpan')

Model berhasil disimpan


# Evaluasi 